In [0]:
from pyspark.sql import functions as F

# 1. DATABRICKS WIDGET FOR PARAMETERIZATION
dbutils.widgets.text("batch_name", "batch_1", "Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Target Batch Selected: {batch_name}")

# 2. INGESTION FUNCTION DEFINITION


def ingest_batch_to_bronze(target_batch):

  print(f"==== Starting Bronze Ingestion for: {target_batch} =====")

  # Base path pointing to your Unity Catalog Volume
  base_vol = f"/Volumes/ecommerce_lakehouse/bronze/raw_files/{target_batch}/"

  # List of all 9 Olist source datasets
  datasets = [
      "olist_orders_dataset",
      "olist_order_items_dataset",
      "olist_order_payments_dataset",
      "olist_order_reviews_dataset",
      "olist_customers_dataset",
      "olist_products_dataset",
      "olist_sellers_dataset",
      "olist_geolocation_dataset",
      "product_category_name_translation",
  ]

  for ds in datasets:
    input_path = f"{base_vol}{ds}/"
    # Target Delta table name in Unity Catalog (stripping '_dataset' for cleaner naming)
    table_name = f"ecommerce_lakehouse.bronze.bronze_{ds.replace('_dataset', '')}"

    try:
      print(f"Processing: {ds} ---> {table_name}")

      # 1. Read raw CSV files
      df = (
          spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(input_path)
      )

      # 2. Add enterprise audit metadata columns
      df_bronze = (
          df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_batch_id", F.lit(target_batch))
      )

      # 3. Determine write mode: Overwrite for batch_1 (initial load), Append for subsequent batches
      write_mode = "overwrite" if target_batch == "batch_1" else "append"

      # 4. Save as Delta Table
      df_bronze.write.format("delta").mode(write_mode).saveAsTable(table_name)

      print(f" -> Successfully wrote to {table_name} using mode='{write_mode}'\n")

    except Exception as e:
      print(f" -> Error processing {ds}: {e}\n")

  print(f"==== Completed Bronze Ingestion for: {target_batch} ==== ")


ingest_batch_to_bronze(batch_name)

Target Batch Selected: batch_1
==== Starting Bronze Ingestion for: batch_1 =====
Processing: olist_orders_dataset ---> ecommerce_lakehouse.bronze.bronze_olist_orders
 -> Successfully wrote to ecommerce_lakehouse.bronze.bronze_olist_orders using mode='overwrite'

Processing: olist_order_items_dataset ---> ecommerce_lakehouse.bronze.bronze_olist_order_items
 -> Successfully wrote to ecommerce_lakehouse.bronze.bronze_olist_order_items using mode='overwrite'

Processing: olist_order_payments_dataset ---> ecommerce_lakehouse.bronze.bronze_olist_order_payments
 -> Successfully wrote to ecommerce_lakehouse.bronze.bronze_olist_order_payments using mode='overwrite'

Processing: olist_order_reviews_dataset ---> ecommerce_lakehouse.bronze.bronze_olist_order_reviews
 -> Successfully wrote to ecommerce_lakehouse.bronze.bronze_olist_order_reviews using mode='overwrite'

Processing: olist_customers_dataset ---> ecommerce_lakehouse.bronze.bronze_olist_customers
 -> Successfully wrote to ecommerce_lake